# Medallion Architecture — Notebook 4: Aggregating Data into the Gold Layer

This is the fourth notebook in the Medallion Architecture tutorial series. Notebooks 1–3 took raw CSV → Bronze (raw, as-is) → Silver (clean, one row per invoice). Here we build the **Gold** layer: aggregated, business-ready tables that answer a specific question.

**The business question this Gold table answers:** *"How are sales, profit, and order volume performing in each country, month by month?"* — the kind of question a sales leader or dashboard asks daily.

We save the result as `demo.demo_sch.gold_country_monthly_sales`.

## Designing the Gold Table First

Unlike Bronze and Silver, a Gold table is designed **starting from the business question, not from the data**. Four decisions up front:

| Design question | Our answer |
|---|---|
| **What business question does this answer?** | "How are sales, profit, and order volume performing in each country, month by month?" |
| **What does each row represent? (the grain)** | One country's sales in one calendar month — e.g. *France, March 2021* |
| **Which columns are grouped (kept as-is)?** | `Country`, `SalesYear`, `SalesMonth` — together they define the grain |
| **Which columns are aggregated?** | `InvoiceNum` → invoice count, `OrderQuantity` → units sold, `SalesAmount` / `TaxAmt` / `TotalProductCost` / `ProfitAmount` → sums, plus derived `avg_order_value` and `pct_of_month_revenue` |

**How business users use this result:**

* Plug it straight into a **dashboard** — country filter + monthly trend line, no SQL needed
* Compare countries and spot best/worst months without touching row-level data
* A tiny table (one row per country-month instead of one row per invoice) — fast to query

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_table = "demo.demo_sch.products_silver"
silver_df = spark.table(silver_table)

print(f"Loaded {silver_df.count():,} invoice rows from {silver_table}")
print("Silver grain: one row per InvoiceNum")
display(silver_df.limit(5))

## Build the Gold Table

We change the grain from **one row per invoice** (Silver) to **one row per country-month** (Gold) using `groupBy` + aggregations, then add derived measures:

* `avg_order_value` — `total_sales / num_invoices` within the country-month
* `pct_of_month_revenue` — a **window function** computing each country's share of worldwide revenue that month

The grouped columns (`Country`, `SalesYear`, `SalesMonth`) survive the aggregation and define each row; everything else is rolled up.

In [0]:
gold_table = "demo.demo_sch.gold_country_monthly_sales"

# GROUPED columns — define the grain: Country + SalesYear + SalesMonth
# AGGREGATED columns — rolled up from all invoices in that country-month
gold_df = (
    silver_df
    .groupBy("Country", "SalesYear", "SalesMonth")
    .agg(
        F.countDistinct("InvoiceNum").alias("num_invoices"),
        F.sum("OrderQuantity").alias("units_sold"),
        F.round(F.sum("SalesAmount"), 2).alias("total_sales"),
        F.round(F.sum("TaxAmt"), 2).alias("total_tax"),
        F.round(F.sum("TotalProductCost"), 2).alias("total_cost"),
        F.round(F.sum("ProfitAmount"), 2).alias("total_profit"),
    )
    # Derived measures on top of the aggregates
    .withColumn("avg_order_value", F.round(F.col("total_sales") / F.col("num_invoices"), 2))
    .withColumn("YearMonth", F.concat_ws("-", F.col("SalesYear"), F.lpad(F.col("SalesMonth").cast("string"), 2, "0")))
)

# Each country's share of that month's worldwide revenue (window over the month)
month_window = Window.partitionBy("SalesYear", "SalesMonth")
gold_df = gold_df.withColumn(
    "pct_of_month_revenue",
    F.round(F.col("total_sales") / F.sum("total_sales").over(month_window) * 100, 1),
)

(
    gold_df
    .select("YearMonth", "Country", "SalesYear", "SalesMonth",
            "num_invoices", "units_sold", "total_sales", "total_tax", "total_cost",
            "total_profit", "avg_order_value", "pct_of_month_revenue")
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_table)
)

gold_rows = gold_df.count()
print(f"Wrote {gold_rows:,} country-month rows to {gold_table}")
print(f"Grain changed: {silver_df.count():,} invoice rows -> {gold_rows:,} country-month rows")

In [0]:
gold_check = spark.table(gold_table)

# A correct grain is verifiable: the number of rows must equal
# the number of distinct combinations of the grouped columns.
grain_cols = ["Country", "SalesYear", "SalesMonth"]
distinct_grain = gold_check.select(*grain_cols).distinct().count()

print(f"Gold rows: {gold_check.count():,}")
print(f"Distinct country-months: {distinct_grain:,}")
print(f"Grain is unique (one row per country-month): {gold_check.count() == distinct_grain}")

gold_check.printSchema()
display(gold_check.orderBy("YearMonth", "Country").limit(12))

In [0]:
# With this one small table, business users answer common questions directly:

# 1) Worldwide revenue trend by month — ready for a dashboard line chart
monthly_trend = (
    gold_check
    .groupBy("YearMonth")
    .agg(F.round(F.sum("total_sales"), 2).alias("monthly_sales"),
         F.round(F.sum("total_profit"), 2).alias("monthly_profit"))
    .orderBy("YearMonth")
)
display(monthly_trend)

# 2) Which countries are most profitable overall?
top_countries = (
    gold_check
    .groupBy("Country")
    .agg(F.round(F.sum("total_profit"), 2).alias("total_profit"))
    .orderBy(F.desc("total_profit"))
)
display(top_countries)

# 3) Each country's single most profitable month
best_month_by_country = (
    gold_check
    .groupBy("Country")
    .agg(F.max(F.struct(F.col("total_profit"), F.col("YearMonth"))).alias("best"))
    .select("Country",
            F.col("best.YearMonth").alias("best_month"),
            F.col("best.total_profit").alias("profit_in_best_month"))
    .orderBy(F.desc("profit_in_best_month"))
)
display(best_month_by_country)

## Recap

* **Designed first, built second** — started from the business question: *sales, profit, and order volume per country, per month*
* **Changed the grain** — one row per invoice (Silver) → one row per country-month (Gold)
* **Grouped columns** define the grain: `Country`, `SalesYear`, `SalesMonth`; everything else is aggregated (invoice count, units, sales/tax/cost/profit sums) or derived (avg order value, share of monthly revenue)
* **Verified the grain** — row count equals distinct country-months, so no duplicated grain rows
* **Business-ready** — small, fast table that dashboards and analysts consume without touching row-level data

**The full journey:** raw CSV → **Bronze** (as-is) → **Silver** (clean, one row per invoice) → **Gold** (aggregated, answers business questions).